In [2]:
import os
import re
import warnings
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

countries = [
    "Angola",
    "Benin",
    "Botswana",
    "Burkina Faso",
    "Burundi",
    "Cameroon",
    "Central African Republic",
    "Chad",
    "Democratic Republic of Congo",
    "Djibouti",
    "Eritrea",
    "Ethiopia",
    "Ghana",
    "Guinea",
    "Ivory Coast",
    "Kenya",
    "Liberia",
    "Madagascar",
    "Mali",
    "Malawi",
    "Mauritania",
    "Mozambique",
    "Namibia",
    "Niger",
    "Nigeria",
    "Republic of the Congo",
    "Rwanda",
    "Senegal",
    "Sierra Leone",
    "Somalia",
    "South Sudan",
    "Sudan",
    "Tanzania",
    "The Gambia",
    "Togo",
    "Uganda",
    "Zambia",
    "Zimbabwe"
]

DRAWS = 2000
TUNE = 2000
TARGET_ACCEPT = 0.95
RANDOM_SEED = 42

OUTPUT_DIR = "bayesian_results"
DETAIL_DIR = os.path.join(OUTPUT_DIR, "country_details")
os.makedirs(DETAIL_DIR, exist_ok=True)


def read_csv_robust(path):
    try:
        df = pd.read_csv(path)
        if len(df.columns) == 1:
            df = pd.read_csv(path, sep=";")
    except Exception:
        df = pd.read_csv(path, sep=";")

    df.columns = df.columns.str.strip()
    return df


def to_numeric_clean(series):
    return pd.to_numeric(
        series.astype(str)
              .str.strip()
              .str.replace(" ", "", regex=False)
              .str.replace(",", ".", regex=False),
        errors="coerce"
    )


def sanitize_filename(name):
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")


def normalize_colname(col):
    return re.sub(r"[^a-z0-9]+", "", str(col).strip().lower())


def find_first_existing_column(df, candidates, df_name):
    normalized_map = {normalize_colname(col): col for col in df.columns}

    for candidate in candidates:
        norm_candidate = normalize_colname(candidate)
        if norm_candidate in normalized_map:
            return normalized_map[norm_candidate]

    raise KeyError(
        f"Column not found in {df_name}. Expected one of {candidates}. "
        f"Available columns: {df.columns.tolist()}"
    )


def find_existing_file(candidates):
    for path in candidates:
        if os.path.exists(path):
            return path
    return None


def build_country_paths(country):
    parasite_candidates = [
        f"Result_{country}_parasite.csv",
        f"{country}_parasite.csv"
    ]

    precip_candidates = [
        f"{country}_mean_precipitation_in_flood_period.csv"
    ]

    hazard_candidates = [
        f"{country}_hazard.csv"
    ]

    parasite_file = find_existing_file(parasite_candidates)
    precip_file = find_existing_file(precip_candidates)
    hazard_file = find_existing_file(hazard_candidates)

    return parasite_file, precip_file, hazard_file


def run_country_model(country):
    parasite_file, precip_file, hazard_file = build_country_paths(country)

    if parasite_file is None or precip_file is None or hazard_file is None:
        return {
            "Country": country,
            "Status": "missing_file",
            "Rows_total": np.nan,
            "Rows_used": np.nan,
            "Rows_skipped": np.nan,
            "Top_driver": np.nan,
            "Top_driver_importance": np.nan,
            "Flood_importance": np.nan,
            "Precipitation_importance": np.nan,
            "Hazard_importance": np.nan,
            "Prob_Flood_strongest": np.nan,
            "Prob_Precipitation_strongest": np.nan,
            "Prob_Hazard_strongest": np.nan,
            "alpha_mean": np.nan,
            "beta_Flood_mean": np.nan,
            "beta_Precipitation_mean": np.nan,
            "beta_Hazard_mean": np.nan,
            "sigma_mean": np.nan,
            "Message": "Missing file"
        }, None

    parasite_df = read_csv_robust(parasite_file)
    precip_df = read_csv_robust(precip_file)
    hazard_df = read_csv_robust(hazard_file)

    parasite_col = find_first_existing_column(
        parasite_df,
        [
            "Global_parasite_count_flood_zone",
            "Global parasite count flood zone"
        ],
        "parasite_df"
    )

    flood_col = find_first_existing_column(
        parasite_df,
        [
            "Flood_exposure",
            "Flood exposure"
        ],
        "parasite_df"
    )

    precip_col = find_first_existing_column(
        precip_df,
        [
            "Precipitation_mean__mm",
            "Precipitation mean mm",
            "Precipitation_mean_mm"
        ],
        "precip_df"
    )

    hazard_col = find_first_existing_column(
        hazard_df,
        [
            "Hazard_(ha)",
            "Hazard (ha)",
            "Hazard_ha",
            "Hazard ha"
        ],
        "hazard_df"
    )

    min_len = min(len(parasite_df), len(precip_df), len(hazard_df))

    parasite_df = parasite_df.iloc[:min_len].copy()
    precip_df = precip_df.iloc[:min_len].copy()
    hazard_df = hazard_df.iloc[:min_len].copy()

    df = pd.DataFrame({
        "Y": parasite_df[parasite_col],
        "Flood": parasite_df[flood_col],
        "Precip": precip_df[precip_col],
        "Hazard": hazard_df[hazard_col]
    })

    for col in ["Y", "Flood", "Precip", "Hazard"]:
        df[col] = to_numeric_clean(df[col])

    rows_total = len(df)

    # Keep only complete rows
    df_valid = df.dropna(subset=["Y", "Flood", "Precip", "Hazard"]).reset_index(drop=True)

    rows_used = len(df_valid)
    rows_skipped = rows_total - rows_used

    # If no usable rows are available, skip the model
    if rows_used == 0:
        return {
            "Country": country,
            "Status": "no_valid_rows",
            "Parasite_file_used": parasite_file,
            "Precip_file_used": precip_file,
            "Hazard_file_used": hazard_file,
            "Rows_total": rows_total,
            "Rows_used": rows_used,
            "Rows_skipped": rows_skipped,
            "Top_driver": np.nan,
            "Top_driver_importance": np.nan,
            "Flood_importance": np.nan,
            "Precipitation_importance": np.nan,
            "Hazard_importance": np.nan,
            "Prob_Flood_strongest": np.nan,
            "Prob_Precipitation_strongest": np.nan,
            "Prob_Hazard_strongest": np.nan,
            "alpha_mean": np.nan,
            "beta_Flood_mean": np.nan,
            "beta_Precipitation_mean": np.nan,
            "beta_Hazard_mean": np.nan,
            "sigma_mean": np.nan,
            "Message": "No complete rows"
        }, None

    # If there is only one row, the Bayesian model does not have enough information
    if rows_used < 2:
        return {
            "Country": country,
            "Status": "too_few_rows",
            "Parasite_file_used": parasite_file,
            "Precip_file_used": precip_file,
            "Hazard_file_used": hazard_file,
            "Rows_total": rows_total,
            "Rows_used": rows_used,
            "Rows_skipped": rows_skipped,
            "Top_driver": np.nan,
            "Top_driver_importance": np.nan,
            "Flood_importance": np.nan,
            "Precipitation_importance": np.nan,
            "Hazard_importance": np.nan,
            "Prob_Flood_strongest": np.nan,
            "Prob_Precipitation_strongest": np.nan,
            "Prob_Hazard_strongest": np.nan,
            "alpha_mean": np.nan,
            "beta_Flood_mean": np.nan,
            "beta_Precipitation_mean": np.nan,
            "beta_Hazard_mean": np.nan,
            "sigma_mean": np.nan,
            "Message": "Too few complete rows"
        }, None

    scaler_X = StandardScaler()
    scaler_Y = StandardScaler()

    X = scaler_X.fit_transform(df_valid[["Flood", "Precip", "Hazard"]])
    Y = scaler_Y.fit_transform(df_valid[["Y"]]).flatten()

    with pm.Model() as model:
        mu_beta = pm.Normal("mu_beta", mu=0, sigma=1)
        sigma_beta = pm.HalfNormal("sigma_beta", sigma=1)

        beta = pm.Normal("beta", mu=mu_beta, sigma=sigma_beta, shape=3)
        alpha = pm.Normal("alpha", mu=0, sigma=1)
        sigma = pm.HalfNormal("sigma", sigma=1)

        mu = alpha + pm.math.dot(X, beta)

        pm.Normal("Y_obs", mu=mu, sigma=sigma, observed=Y)

        trace = pm.sample(
            draws=DRAWS,
            tune=TUNE,
            target_accept=TARGET_ACCEPT,
            random_seed=RANDOM_SEED,
            progressbar=False,
            return_inferencedata=True,
            cores=1,
            chains=4
        )

    summary = az.summary(trace, var_names=["alpha", "beta", "sigma"]).reset_index()
    summary = summary.rename(columns={"index": "Parameter"})
    summary.insert(0, "Country", country)

    beta_samples = trace.posterior["beta"].stack(sample=("chain", "draw")).values
    if beta_samples.shape[0] != 3:
        beta_samples = beta_samples.T

    names = ["Flood_exposure", "Precipitation", "Hazard"]

    importance = np.mean(np.abs(beta_samples), axis=1)
    order = np.argsort(importance)[::-1]

    winner = np.argmax(np.abs(beta_samples), axis=0)
    strongest_probs = [float(np.mean(winner == i)) for i in range(3)]

    result_row = {
        "Country": country,
        "Status": "ok",
        "Parasite_file_used": parasite_file,
        "Precip_file_used": precip_file,
        "Hazard_file_used": hazard_file,
        "Rows_total": rows_total,
        "Rows_used": rows_used,
        "Rows_skipped": rows_skipped,
        "Top_driver": names[order[0]],
        "Top_driver_importance": float(importance[order[0]]),
        "Flood_importance": float(importance[0]),
        "Precipitation_importance": float(importance[1]),
        "Hazard_importance": float(importance[2]),
        "Prob_Flood_strongest": strongest_probs[0],
        "Prob_Precipitation_strongest": strongest_probs[1],
        "Prob_Hazard_strongest": strongest_probs[2],
        "alpha_mean": float(summary.loc[summary["Parameter"] == "alpha", "mean"].iloc[0]),
        "beta_Flood_mean": float(summary.loc[summary["Parameter"] == "beta[0]", "mean"].iloc[0]),
        "beta_Precipitation_mean": float(summary.loc[summary["Parameter"] == "beta[1]", "mean"].iloc[0]),
        "beta_Hazard_mean": float(summary.loc[summary["Parameter"] == "beta[2]", "mean"].iloc[0]),
        "sigma_mean": float(summary.loc[summary["Parameter"] == "sigma", "mean"].iloc[0]),
        "Message": "ok"
    }

    return result_row, summary


all_results = []
all_summaries = []

for country in countries:
    try:
        result_row, summary_df = run_country_model(country)
        all_results.append(result_row)

        if summary_df is not None:
            all_summaries.append(summary_df)
            safe_country = sanitize_filename(country)
            summary_path = os.path.join(DETAIL_DIR, f"{safe_country}_model_summary.csv")
            summary_df.to_csv(summary_path, index=False)

    except Exception as e:
        all_results.append({
            "Country": country,
            "Status": "error",
            "Rows_total": np.nan,
            "Rows_used": np.nan,
            "Rows_skipped": np.nan,
            "Top_driver": np.nan,
            "Top_driver_importance": np.nan,
            "Flood_importance": np.nan,
            "Precipitation_importance": np.nan,
            "Hazard_importance": np.nan,
            "Prob_Flood_strongest": np.nan,
            "Prob_Precipitation_strongest": np.nan,
            "Prob_Hazard_strongest": np.nan,
            "alpha_mean": np.nan,
            "beta_Flood_mean": np.nan,
            "beta_Precipitation_mean": np.nan,
            "beta_Hazard_mean": np.nan,
            "sigma_mean": np.nan,
            "Message": str(e)
        })

results_df = pd.DataFrame(all_results)
results_path = os.path.join(OUTPUT_DIR, "bayesian_results_all_countries.csv")
results_df.to_csv(results_path, index=False)

if all_summaries:
    summaries_df = pd.concat(all_summaries, ignore_index=True)
    summaries_path = os.path.join(OUTPUT_DIR, "bayesian_coeff_summary_all_countries.csv")
    summaries_df.to_csv(summaries_path, index=False)

print(results_df)
print(f"\nSaved: {results_path}")

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (4 chains in 1 job)
NUTS: [mu_beta, sigma_beta, beta, alpha, sigma]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 20 seconds.
There were 79 divergences after tuning. Increase `target_accept` or reparameterize.
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (4 chains in 1 job)
NUTS: [mu_beta, sigma_beta, beta, alpha, sigma]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 26 seconds.
There were 63 divergences after tuning. Increase `target_accept` or reparameterize.
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (4 chains in 1 job)
NUTS: [mu_beta, sigma_beta, beta, alpha, sigma]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 19 seconds.
There were 4 divergences after tuning. Increase `target_accept` or reparameterize.
Initializing NUTS using jitter+adapt_d

                         Country Status  \
0                         Angola     ok   
1                          Benin     ok   
2                       Botswana     ok   
3                   Burkina Faso     ok   
4                        Burundi     ok   
5                       Cameroon     ok   
6       Central African Republic     ok   
7                           Chad     ok   
8   Democratic Republic of Congo     ok   
9                       Djibouti     ok   
10                       Eritrea     ok   
11                      Ethiopia     ok   
12                         Ghana     ok   
13                        Guinea     ok   
14                   Ivory Coast     ok   
15                         Kenya     ok   
16                       Liberia     ok   
17                    Madagascar     ok   
18                          Mali     ok   
19                        Malawi     ok   
20                    Mauritania     ok   
21                    Mozambique     ok   
22         